In [ ]:
# Configure model, metadata and import libraries
import pandas as pd
import json
import requests
import time
import google.generativeai as genai
from google.colab import userdata
import warnings
warnings.filterwarnings('ignore')

# Set Gemini API Key
# Replace 'Your_api_key_name' with the name in your Colab Notebook
api_key = userdata.get('Your_api_key_name')
genai.configure(api_key=api_key)

# Set Gemini Model
model='gemini-2.5-flash'

# Retrieve Input data and Human evaluated data from GitHub
data_human_url = 'https://raw.githubusercontent.com/FootlooseNFree/GetYourGuide/refs/heads/main/GetYourGuide_CaseStudy.csv'
data_human = pd.read_csv(data_human_url)

# Retrieve System Prompt from GitHub
SYSTEM_PROMPT_URL = 'https://raw.githubusercontent.com/FootlooseNFree/GetYourGuide/refs/heads/main/system_prompt_de.txt'
response = requests.get(SYSTEM_PROMPT_URL)
SYSTEM_PROMPT = response.text

# Configure Model
generation_config = {
    'temperature': 0.0,
    'response_mime_type': 'application/json'
}

model = genai.GenerativeModel(
    model_name=model,
    system_instruction=SYSTEM_PROMPT,
    generation_config=generation_config
)

In [ ]:
# Define evaluation function
def evaluate_activity_title(source_en, target_de):
    user_prompt = f'Source Text (en-US): {source_en}\nTarget Text (de-DE): {target_de}'
    try:
        response = model.generate_content(user_prompt)
        return json.loads(response.text)
    except Exception as e:
        print(f'Error while evaluating: {source_en} --> {e}')
        return None

In [ ]:
# Perform evaluation
model_results = []
for idx, row in data_human.iterrows():
    print(f'Evaluating - {row['text_en']}')
    result = evaluate_activity_title(row['text_en'], row['text_de'])
    model_results.append(result if result else {})

    if idx < len(data_human) - 1:
        print('Sleeping for 90 seconds...')
        time.sleep(90)

# Save the results
data_model = pd.DataFrame(model_results)
data_human['model_evaluation_error_categories'] = data_model[
        'model_evaluation_error_categories'
        ].apply(lambda x: ",".join(x) if isinstance(x, list) else "")
data_human['model_clickability_score'] = data_model['model_clickability_score']
data_human['model_reasoning'] = data_model['model_reasoning']
data_human['model_localization'] = data_model['model_localization']

# Save final evaluation as csv
data_human.to_csv('GetYourGuide_CaseStudy_Gemini_Evaluated.csv', index=False)
print('Data evaluated and written successfully')